<a href="https://colab.research.google.com/github/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/notebooks/02_Working_with_Data_in_Python/02_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐼 Working with Data in Python — pandas

---

An omics experiment ends as a table. The instrument and the processing pipeline hand you a
rectangle of numbers with some labels around the edges, and everything that happens
afterwards — filtering, normalisation, testing, plotting, integration — is a transformation
of that rectangle. **pandas** is the library that lets us write those transformations in a
few readable lines instead of a nested loop, and it is the floor that every other tool in
this course stands on.

This session is deliberately not a tour of the API. It is an hour spent on **the tables we
use for the rest of the course**: the clinical metadata of 45 septic patients, a matrix of
1 458 protein groups and a matrix of 1 073 metabolites measured on the same serum samples.
Every example below is a real column with a real meaning, and by this afternoon — when we
plot these data and then run a full differential-abundance analysis on them — the
operations in this notebook are the ones doing the work.

Three ideas earn most of the hour: getting a table **oriented** the way statistics expects
it, moving between **wide** and **long** shape, and being honest about **missing values**.
The rest is vocabulary, and vocabulary you can look up.

### What you will be able to do afterwards

1. Build a `DataFrame` by hand, and read one from a file or straight from a URL.
2. Say what the index and the columns are for, and set an index that means something.
3. Select and filter rows and columns with `loc`, `iloc` and boolean masks.
4. Add and remove columns, and reshape between wide and long with `melt` and `pivot`.
5. Join two tables on a shared identifier with `merge`, choosing `how` deliberately.
6. Summarise a table with `describe`, `value_counts` and `groupby`.
7. Find the missing values in an omics matrix and argue for what to do about them.

## 1. The data we learn on

pandas handles tabular data with columns of different types — an Excel sheet, essentially —
as well as time series and more or less anything observational you can lay out in rows and
columns. We meet it through one dataset rather than through toy examples, because the
awkward parts of pandas are exactly the parts real data force you to learn.

Everything below uses serum samples from **45 septic patients, 15 per group**:

| group | meaning |
| :--- | :--- |
| `Con` | sepsis with **negative** blood cultures |
| `CSKP` | carbapenem-**susceptible** *Klebsiella pneumoniae* sepsis |
| `CRKP` | carbapenem-**resistant** *Klebsiella pneumoniae* sepsis |

For each patient we have clinical metadata, a proteomics matrix of 1 458 protein groups
(diaPASEF / DIA-NN) and a metabolomics matrix of 1 073 named metabolites (targeted MRM).
The same three tables reappear in the visualisation session after lunch, in the proteomics
analysis this afternoon and in the multi-omics integration on Day 3, so time spent learning
their quirks now is time you do not spend debugging tomorrow.

Two things about the sample names are worth knowing before we touch the data. First, the
sample identifiers of the carbapenem-susceptible group start with `KP` (`KP1` … `KP15`) even
though the group is called `CSKP` in the metadata; this inconsistency comes from the
original study and we simply have to live with it. Second, the columns named `QC_pool*`
(proteomics) or `QC*` (metabolomics) are **pooled quality-control injections, not
patients**, so they must be excluded whenever we compute anything per patient. Both traps
are the reason we select sample columns by an explicit rule below rather than by counting
columns from the left.

### Where the data live

Nothing is downloaded by hand. Every file sits in the course repository on GitHub, and
pandas reads directly from a URL, so the notebook behaves identically on Colab, on a laptop
and on a cluster — no data folder to set up, and no chance of quietly analysing last week's
copy. We assemble the URLs once, here at the top, and refer to them by name from then on.

> ⚙️ Building paths from a `BASE_URL` constant instead of pasting a long URL into every call
> is a small habit with a large payoff. Switching to a different branch of the repository, or
> to a local checkout, becomes a one-line change rather than a search-and-replace through a
> notebook — and the reader can see at a glance exactly which files the analysis depends on.

In [ ]:
import pandas as pd

# All course data sets live in the course GitHub repository. We build the URLs
# once here and reuse them throughout the notebook.
COURSE_REPO = "Multiomics-Analytics-Group/course_multi-omics_analysis"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{COURSE_REPO}/{BRANCH}"

# Clinical metadata, one row per patient
METADATA_URL = f"{BASE_URL}/metadata/sample_metadata.tsv"
# Omics matrices, one row per feature and one column per sample
PROTEINS_URL = f"{BASE_URL}/proteomics/data/protein_groups_matrix.tsv"
METABOLITES_URL = f"{BASE_URL}/metabolomics/data/metabolite_matrix.tsv"
# Annotation and published results (used in the exercises and in later notebooks)
METABOLITE_ANNOTATION_URL = f"{BASE_URL}/metabolomics/data/metabolite_annotation.tsv"
PUBLISHED_DEPS_URL = f"{BASE_URL}/proteomics/data/published_deps.tsv"

## 2. Creating a data frame

A `DataFrame` is a table: named columns, each with its own type, sharing one row index. You
will usually read one from a file, but building one by hand is worth knowing — it is how you
write a small test case, assemble a results table inside a loop, or type out the four rows
you need to check that a function does what you think.

There are three routes to the same table, and which one you take depends on how the data
reach you: **column by column** (a dictionary of lists), **row by row** (a list of lists,
with the column names supplied separately), or **record by record** (a list of
dictionaries). All three produce an identical table; none is more correct than the others.

In [ ]:
# A tiny table by hand: one row per patient, one column per variable
df = pd.DataFrame({
    'c_reactive_protein': [1.0, 3.0, 96.0, 145.0],   # mg/L
    'age': [55, 30, 59, 71],                         # years
    'group': ["Con", "Con", "CSKP", "CRKP"]
})

In [ ]:
df

In [ ]:
# The same table from a list of rows (a list of lists)
data = [
    [1.0, 55, "Con"],
    [3.0, 30, "Con"],
    [96.0, 59, "CSKP"],
    [145.0, 71, "CRKP"]
]

df = pd.DataFrame(data, columns=['c_reactive_protein', 'age', 'group'])
df

### From a `list` of `dictionaries`

Each dictionary is one row, and its keys become the columns. This is the shape data arrive
in when they come back from a web API, or when you append one result per iteration of a
loop. pandas lines the keys up for you: a record that is missing a key gets a `NaN` in that
column rather than an error — convenient, and something to be aware of, because a typo in a
key name produces an extra half-empty column instead of a complaint.

In [ ]:
data = [
    {'c_reactive_protein': 1.0, 'age': 55, 'group': "Con"},
    {'c_reactive_protein': 3.0, 'age': 30, 'group': "Con"},
    {'c_reactive_protein': 96.0, 'age': 59, 'group': "CSKP"},
    {'c_reactive_protein': 145.0, 'age': 71, 'group': "CRKP"}
]

df = pd.DataFrame(data)
df

### Creating an empty `DataFrame`

You can also start from an empty table and attach columns one at a time. It reads perfectly
well for three columns of four values. Do not do the row-wise equivalent in a loop over
thousands of records: pandas rebuilds the whole table on every assignment, so what looks
like an innocent append turns into minutes of copying. Collect the rows in a plain Python
list and call `pd.DataFrame` once at the end.

In [ ]:
df = pd.DataFrame()
df['c_reactive_protein'] = [1.0, 3.0, 96.0, 145.0]
df['age'] = [55, 30, 59, 71]
df['group'] = ["Con", "Con", "CSKP", "CRKP"]

df

### ✋ Exercise 1

Create the following data frame. It holds the median value of two clinical markers in two of
the patient groups — one row per group/marker combination. Use whichever of the three routes
above you find clearest, and pay attention to the shape while you type it: one row per
*measurement*, with the group and the marker named in columns of their own. That is **long**
format, and section 7 is about why it matters.

|  | Group | Marker | Median |
| ---| :--: | :----:  | :--: |
| 0  | Con  | procalcitonin       | 0.07 |
| 1  | CRKP | procalcitonin       | 2.33 |
| 2  | Con  | c_reactive_protein  | 5.00 |
| 3  | CRKP | c_reactive_protein  | 105.00 |

## 3. Reading and writing files

pandas reads almost anything, and the function is nearly always named after the format.
What actually decides whether the table comes out right is a handful of arguments —
the separator, which row holds the header, what counts as a missing value — because getting
them wrong rarely raises an error. It gives you a table that looks plausible and is wrong,
which is far more expensive.

| File Type | Function Name |
| :----:    |  :---:  |
| Excel | `pd.read_excel` |
| CSV, TSV | `pd.read_csv` |
| H5, HDF, HDF5 | `pd.read_hdf` |
| JSON  | `pd.read_json` |
| SQL | `pd.read_sql_table` |

Each has a `to_` counterpart for writing: `to_csv`, `to_excel`, `to_hdf`, and so on.

### Reading data

We read the data directly from GitHub. To get such a link yourself, open the file on GitHub
and use the **Raw** option: that serves the file as plain text and gives you a URL pandas
can read with no further work. It is worth preferring this over a downloaded copy whenever
you can — a URL is a statement about *which version* of the data you analysed, and a file
called `data_final_v3.tsv` on your desktop is not.

The course data are in this repository:

https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis

and we assembled the URLs in the cell at the top of the notebook, e.g. `METADATA_URL` points
at `metadata/sample_metadata.tsv`.

All course files are **tab-separated**, so `sep='\t'` must be passed to `pd.read_csv`.
Forget it and pandas reads each line as a single column with tabs inside it — a one-column
table is the classic symptom of a forgotten separator.

**Comma separated/Tab separated files (.csv, .tsv)**

In [ ]:
metadata = pd.read_csv(METADATA_URL, sep='\t')
metadata.head()

**Excel files (.xls, .xlsx)**

`pd.read_excel` needs a sheet name when the workbook has more than one, and an extra
package (`openpyxl`) installed. Spreadsheets are how clinical data usually arrive, so you
will use this — but do not *store* analysis inputs as `.xlsx` if you can avoid it. Excel
reformats gene symbols into dates, drops leading zeros from identifiers and hides
provenance in cell formatting; plain tab-separated text does none of those things.

In [ ]:
# The same table stored as a spreadsheet would be read like this:
# metadata = pd.read_excel("sample_metadata.xlsx", sheet_name='metadata')

### Writing data

Writing is the mirror image. The argument to think about is `index=False`: the row numbers
0…44 are an artefact of how pandas loaded the table, not part of the data, and writing them
out gives the next reader a mystery column called `Unnamed: 0`. Write the index only when it
carries meaning — a sample identifier, a protein group — and then give it a name.

On Colab, files written this way land on the runtime's temporary disk and disappear when the
session is recycled. Download anything you want to keep, or write it to a mounted Drive as
described in section 4.

**Comma separated/Tab separated files (.csv, .tsv)**

In [ ]:
# Write the table back out as a tab-separated file.
# index=False leaves out the row numbers, which are not part of the data.
metadata.to_csv("sample_metadata_copy.tsv", sep='\t', index=False)

**Excel files (.xls, .xlsx)**

In [ ]:
# metadata.to_excel("sample_metadata.xlsx", sheet_name='metadata', index=False)

## 4. Accessing data in Google Drive

Reading straight from a URL, as we just did, is the simplest way to get data into a
notebook; mounting your Google Drive is the alternative, and it is what you will want for
your own project data and for anything you need to keep between sessions. None of the course
notebooks need it — they all read from the repository over HTTPS, so there is nothing to
mount — so treat this section as Colab housekeeping to come back to later.

1. Run the code below. It gives you an authentication link so that you can access your
   Google Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
import os
# list current directory
os.listdir()

Once mounted, your Drive appears as an ordinary folder under `/content/gdrive`: you can
browse it in the file panel on the left and read from it with exactly the same `read_csv`
call as above, giving a path instead of a URL.
![image.png](https://i0.wp.com/neptune.ai/wp-content/uploads/2022/10/colab-drive.png?ssl=1)

## 5. Indexing — and which way round a table goes

Every data frame has an **index**: the labels that identify its rows. By default that is the
integers 0, 1, 2 …, which carry no information whatsoever. Whenever a column already
identifies the row — here `sample_id` — promoting it to the index pays for itself
immediately. `loc` can then address a patient by name; joins and alignment between tables
happen on the identifier rather than on position; and a table that gets sorted or filtered
takes its labels with it.

That last point is the real argument. Row order is the thing to distrust: two tables that
line up today line up right up until someone sorts one of them. `.loc` selects by **label**,
`.iloc` by **position**, and the difference is worth internalising early.

### Which way round should the table be?

Statistics packages — `scikit-learn`, `statsmodels`, `acore` — expect **samples in rows and
features in columns**, because a sample is an observation and a protein is a variable. Mass
spectrometry software outputs the transpose: one row per protein group, one column per
sample. That is how `protein_groups_matrix.tsv` is stored, and how you will meet it in
section 7. The clinical metadata, by contrast, already arrives the statistician's way round
— one row per patient — which is why the two tables feel different to work with.

> ⚙️ Neither orientation is wrong; what is wrong is being unsure which one you are holding.
> In the proteomics analysis notebook this afternoon we transpose once, immediately after
> loading — `data = proteins_wide.set_index("protein_group")[patient_ids].T` — and never think
> about it again. Transposing early avoids a great deal of confusion later.

In [ ]:
metadata.index

In [ ]:
# The row numbers 0...44 carry no meaning. The sample identifier does,
# so we can use it as the index of the table.
metadata_by_sample = metadata.set_index("sample_id")
metadata_by_sample.head()

In [ ]:
# .loc selects by label, .iloc selects by position
print("CRP of patient CRKP1:", metadata_by_sample.loc["CRKP1", "c_reactive_protein"])
print("First row, first column:", metadata_by_sample.iloc[0, 0])

# Both accept lists, so we can pick several rows and columns at once
metadata_by_sample.loc[["Con1", "KP1", "CRKP1"], ["group", "age", "c_reactive_protein"]]

### Structural properties

Four attributes tell you almost everything about a table you have just loaded: `shape`
(rows × columns), `dtypes` (what pandas decided each column holds), `index` and `columns`.

`dtypes` is the one to read properly. A column you expect to be numeric that comes back as
`object` means pandas found something non-numeric in it — a stray `"NA"`, a comma used as a
decimal separator, a footnote marker, a value recorded as `"<0.05"`. Every arithmetic
operation afterwards will then either fail loudly or, worse, concatenate strings and look
like it worked.

In [ ]:
metadata.shape

In [ ]:
metadata.dtypes

In [ ]:
metadata.index

In [ ]:
metadata.columns

In [ ]:
metadata['c_reactive_protein']

### Inspecting tables

`head` and `tail` show you the two ends — always look at both, because sorted files and
appended blocks of QC samples hide at the bottom. `info` reports the column types together
with how many values are **not** missing, which is the quickest way to spot a column that is
half empty. `describe` summarises the numeric columns. Run all three on any table before you
trust a single number that comes out of it.

In [ ]:
metadata.head()

In [ ]:
metadata.tail(3)

In [ ]:
# info() summarises the columns, their types and how many values are not missing
metadata.info()

# describe() summarises the numeric columns
metadata[['age', 'bmi', 'c_reactive_protein', 'procalcitonin']].describe()

### Selection and filtering

Filtering in pandas is done with **boolean masks**, not with loops. A comparison applied to
a column returns a Series of `True`/`False` of the same length, and indexing the table with
that Series keeps the `True` rows:

```python
metadata['c_reactive_protein'] > 100   # a Series: True where the value is above 100
```

To filter, put the condition in square brackets:

```python
selected_rows = metadata[metadata['c_reactive_protein'] > 100]
```

or, equivalently and more explicitly:

```python
selected_rows = metadata.loc[metadata['c_reactive_protein'] > 100]
```

Masks combine with `&` (and), `|` (or) and `~` (not), and **each condition needs its own
parentheses** — `&` binds more tightly than `>` in Python, so leaving them out produces a
thoroughly unhelpful error about ambiguous truth values. Learn it by the parenthesis rather
than by the traceback.

One thing to keep in mind for section 10: a comparison against `NaN` is always `False`. A
patient with no CRP recorded is silently dropped by `CRP > 100` **and** by `CRP <= 100`, so
the two filters do not add up to the whole cohort. Missing values do not announce
themselves; they just quietly shrink your n.

In [ ]:
# A boolean mask: True for every carbapenem-resistant K. pneumoniae patient
mask = metadata['group'] == "CRKP"
print(mask.head())

crkp = metadata[mask]
print("CRKP patients:", crkp.shape)

# Masks can be combined with & (and) and | (or).
# Each condition needs its own parentheses.
severe_infected = metadata[(metadata['c_reactive_protein'] > 100) & (metadata['group'] != "Con")]
severe_infected[['sample_id', 'group', 'age', 'c_reactive_protein', 'procalcitonin']]

## 6. Modifying a table

Adding and removing columns is where analysis code actually spends its time: a matrix
arrives, and we derive from it the few numbers we intend to test or plot. Two habits keep
this safe. Work on a `copy()` whenever you want the original to survive — pandas is entitled
to hand you a *view* onto another table, and writing to a view is the source of the
`SettingWithCopyWarning` that everybody has seen and nobody reads. And derive new columns
with vectorised operations over whole columns instead of looping over rows: it is shorter,
far faster, and it keeps missing values behaving like missing values.

### Removing a column

`del clinical['group_order']` removes a column in place; `clinical.drop(columns=[...])` does
the same but returns a new table, which is usually what you want in the middle of a chain of
operations. `group_order` encodes Con → CSKP → CRKP as 0, 1, 2 so that plots come out in the
clinically sensible order rather than alphabetically. It is a plotting aid, not a
measurement, and it has no business being fed to a statistical model — which is precisely
why we drop it here.

In [ ]:
clinical = metadata.copy()    # work on a copy so the original table stays intact
del clinical['group_order']   # a helper column used only for plotting order

In [ ]:
clinical.head()

### Creating a new column

The comorbidities are stored as one 0/1 flag per condition — a common encoding, and an
awkward one to summarise, since "how ill was this patient to begin with?" is spread across
six columns. Summing along the rows collapses them into a single count per patient, which is
the kind of derived variable you will later want to check for confounding.

The argument to get right is `axis`, and the way to remember it is: `axis=1` means "across
the columns, once per row".

In [ ]:
# The comorbidities are stored as 0/1 flags, one column each.
# Summing them along the rows (axis=1) gives the number of comorbidities per patient.
comorbidities = ['diabetes', 'heart_disease', 'copd',
                 'liver_disease', 'cerebrovascular_disease', 'kidney_disease']

clinical['n_comorbidities'] = clinical[comorbidities].sum(axis=1)
clinical[['sample_id', 'group'] + comorbidities + ['n_comorbidities']].head()

## 7. Wide and long format: `melt` and `pivot`

This is the most practically useful idea in the session, and the one people take longest to
make peace with.

An omics matrix is **wide**: one row per feature, one column per sample. Our protein table
is 1 458 rows by 45 patient columns. That is compact, it is how the data come out of DIA-NN,
and it is the right shape for matrix operations — correlations, PCA, clustering, heat maps,
anything that treats the whole rectangle at once.

Almost every plotting and statistics function wants the opposite. In **long** format there
is one row per *measurement*: the protein and the sample appear as ordinary columns, and the
intensity sits in a column of its own. The same 1 458 × 45 numbers become 65 610 rows and a
handful of columns. It is bulkier — the protein name is repeated 45 times — but now "colour
by group", "one panel per protein" or "test within each group" is simply another column to
point a function at. That is what `seaborn` and `plotly` are built around, and it is why the
visualisation session after lunch starts by melting a matrix.

| shape | one row is | good for |
| :--- | :--- | :--- |
| **wide** | one feature, measured in every sample | matrix maths, storage, PCA, clustering, heat maps |
| **long** | one feature in one sample | plotting, grouped statistics, joining on sample or feature |

`melt` goes wide → long; `pivot` goes long → wide. Neither invents nor discards information
— they are two spellings of the same table — and being able to switch on demand is most of
what "data wrangling" means in practice.

In [ ]:
proteins = pd.read_csv(PROTEINS_URL, sep='\t')
print("protein matrix:", proteins.shape)
proteins.iloc[:5, :8]

Note the anatomy of the wide matrix: the first four columns *describe the protein group*
(identifier, protein names, genes, description) and every remaining column is a sample. The
last three, `QC_pool1`–`QC_pool3`, are the pooled quality-control injections — the same
material run three times to check instrument stability, not patients. Selecting the patient
columns by an explicit rule, rather than by slicing at a fixed position, is what stops a QC
injection from ending up in a group mean.

In [ ]:
info_columns = ['protein_group', 'protein_names', 'genes', 'description']
qc_columns = [c for c in proteins.columns if c.startswith("QC")]
sample_columns = [c for c in proteins.columns
                  if c not in info_columns and c not in qc_columns]

print("annotation columns:", info_columns)
print("QC columns:", qc_columns)
print("patient columns:", len(sample_columns), "->", sample_columns[:3], "...", sample_columns[-3:])

In [ ]:
# Wide -> long
proteins_long = proteins.melt(
    id_vars=['protein_group', 'genes'],
    value_vars=sample_columns,
    var_name='sample_id',
    value_name='intensity',
)
print(proteins_long.shape, "= 1458 proteins x 45 patients")
proteins_long.head()

Look carefully at what `melt` kept and what it threw away. `id_vars` are the columns that
identify the feature; they are carried down and repeated on every row. `value_vars` are the
columns that get folded into the new `sample_id` column. Anything named in neither is simply
gone — so if you want the gene symbol available for labelling a plot later, it has to be an
`id_var` now.

`pivot` reverses the operation, and it is strict about it: the index/column pair must be
unique. If it complains about duplicate entries, you have more than one measurement per
(protein, sample) and you have to decide what that means. `pivot_table` will average them
for you, which is convenient and occasionally a good way to hide a real problem — two
technical replicates and a botched merge look identical at that point.

In [ ]:
# Long -> wide again
proteins_wide = proteins_long.pivot(index='protein_group',
                                    columns='sample_id',
                                    values='intensity')
print(proteins_wide.shape)
proteins_wide.iloc[:5, :5]

## 8. Combining tables: `merge`

The long protein table knows the sample identifier but not which group the patient belongs
to; the metadata knows the group but nothing about proteins. `merge` joins them on the
shared column `sample_id` — the same idea as a `JOIN` in SQL. Keeping measurements and
clinical annotation in separate files and joining them on an explicit identifier is a habit
worth acquiring: encoding group membership in file names or in column order is how cohorts
get scrambled, and the scrambling is invisible afterwards.

The argument that decides what a merge *means* is `how`:

| `how` | keeps |
| :--- | :--- |
| `'left'` | every row of the left table; unmatched right-hand columns become `NaN` |
| `'inner'` | only the rows that match on both sides |
| `'outer'` | everything from both sides, filling gaps with `NaN` |

The same operation works on features instead of samples: the metabolomics matrix
(`METABOLITES_URL`) can be merged with its annotation table (`METABOLITE_ANNOTATION_URL`) on
the column `metabolite` to attach the compound class and the KEGG/HMDB identifiers. Only 408
of the 1 073 measured metabolites carry an annotation, so `how` is a real decision there:
`how='left'` keeps all 1 073 and leaves the annotation columns empty for the rest, while
`how='inner'` reduces your dataset to 408 metabolites without saying so.

Print the shape before and after every merge. A join that quietly loses two-thirds of your
data looks exactly like a join that worked.

In [ ]:
proteins_annotated = proteins_long.merge(
    metadata[['sample_id', 'group', 'group_label', 'age', 'sex']],
    on='sample_id',
    how='left',      # keep every row of the left table
)
print(proteins_annotated.shape)
print("rows without a matching sample:", proteins_annotated['group'].isna().sum())
proteins_annotated.head()

## 9. Descriptive statistics

Before any test, describe. The individual functions below are all one-liners and none of
them will surprise you; the reflex they are meant to build is looking at a distribution
*before* summarising it. A mean CRP for the whole cohort averages three groups whose typical
values differ by an order of magnitude, and reports a number that describes no one.
`groupby` is what turns that single misleading figure into three informative ones — split
the table by a column, compute the statistic within each part, put the results back
together.

```python
metadata.describe()                     # describes the numeric columns
metadata['age'].describe()              # describes one particular column
metadata['bmi'].count()                 # counts the non-missing values
metadata['group'].nunique()             # counts the unique values
metadata['group'].unique()              # returns the unique values of the column
metadata['group'].value_counts()        # returns the unique values and how often each occurs

metadata['c_reactive_protein'].max()
metadata['c_reactive_protein'].mean()
metadata['c_reactive_protein'][metadata['group'] == 'CRKP'].sum()
```

### Computing the mean of a column

Selecting one column gives a `Series`, and a Series carries the summary statistics as
methods. They all skip missing values by default — convenient, and worth remembering when
you compare two columns whose amounts of missingness differ, because the two means are then
computed over different sets of patients.

In [ ]:
metadata['c_reactive_protein'].mean()

### Computing the sum of a column

The comorbidity flags are 0/1, so summing a column counts patients rather than adding up
quantities. This is a small example of a general point: what a statistic *means* depends
entirely on how the column was encoded, and pandas will happily compute the mean of a
patient identifier if you let it.

In [ ]:
# The comorbidity flags are 0/1, so the sum is the number of patients with diabetes
metadata['diabetes'].sum()

In [ ]:
metadata['age'].min()

In [ ]:
metadata['age'].max()

In [ ]:
# How many patients are there per group?
print(metadata['group'].value_counts())

# sort_values orders the rows; here the patients with the highest CRP come first
print(metadata.sort_values('c_reactive_protein', ascending=False)
              [['sample_id', 'group', 'c_reactive_protein']].head())

# groupby splits the table by group and computes the statistic within each group
metadata.groupby('group')[['age', 'bmi', 'c_reactive_protein', 'procalcitonin']].mean()

### ✋ Exercise 2

Everything so far has been demonstrated for you. These five questions ask you to put the
pieces together on the real matrix. Read each one and decide which operation it needs before
you start typing — and when a result surprises you, print `.shape` at every step, because
most pandas confusion turns out to be a shape you did not expect.

```python
proteins = pd.read_csv(PROTEINS_URL, sep='\t')
metadata = pd.read_csv(METADATA_URL, sep='\t')
```

1. Show the first and the last 5 lines of the protein matrix. What is at the bottom that is
   not at the top?

2. Create a new table that contains only the columns: `protein_group`, `genes` and the 15
   `CRKP` samples. Select them by a rule rather than by typing out fifteen names.

3. Create a new long-format table with `protein_group`, `genes`, `sample_id`, `intensity`
   and `group`, for the infected patients only (`CSKP` and `CRKP`). Remember that the CSKP
   sample identifiers begin with `KP`.

4. Which patient has the highest C-reactive protein value? And which protein group has the
   highest mean intensity across patients? Would you expect the second answer to be
   biologically interesting, or is it telling you something about serum?

5. How many patients are there per group, and for how many protein groups is a gene name
   reported? What should happen to the ones without a gene name — and does it depend on what
   you plan to do next?

## 10. Missing values

About **27 %** of the protein matrix is `NaN`. That is not a data-quality failure; it is what
label-free mass spectrometry looks like, and how you deal with it will change your results
more than most of the decisions you agonise over later.

The tools are simple. First, finding and removing:

| method | description |
| ---:  | :---- |
| **`isna()`** | returns True for every NaN |
| **`notna()`** | returns False for every NaN |
| **`dropna()`** | returns only the rows that contain no NaNs |

and then, if you need a complete matrix, filling:

| method | description |
| ----: |  :---- |
| **`fillna()`** | replaces NaNs with a given value |
| **`ffill()`** | replaces NaNs with the previous non-NaN value |
| **`bfill()`** | replaces NaNs with the next non-NaN value |
| **`interpolate()`** | interpolates between the previous and the following values |

The decision is not simple at all.

> 🧬 **`NaN` is not zero — and in proteomics it is not random either.** A protein is usually
> absent from a run because its concentration was *below the detection limit*: it is missing
> because it is low, not because the instrument looked away. Missingness therefore carries
> information about abundance, which is what statisticians mean by **missing not at random**.
> Fill those `NaN`s with 0 and you assert the protein was truly absent, on a log scale an
> impossible claim. Fill them with the row mean and you do something worse — you take the
> proteins the instrument found least and tell your statistics they were perfectly typical.

There is no free option here, only a choice of what to pay. Dropping every protein with a
missing value gives an honest complete matrix and discards most of the proteome. Requiring a
protein in, say, 70 % of patients keeps far more and still leaves gaps to fill. Imputing
gives you a full matrix and quiet over-confidence: any protein whose significance rests
mostly on imputed values is a candidate artefact. The cells below run each of these on our
matrix so that you can see what each one costs.

In [ ]:
protein_values = proteins[sample_columns]

print("Missing values in total:", int(protein_values.isna().sum().sum()),
      "out of", protein_values.size)
print("Fraction missing: {:.1%}".format(protein_values.isna().to_numpy().mean()))

# How many patients is each protein missing in?
missing_per_protein = protein_values.isna().sum(axis=1)
missing_per_protein.value_counts().sort_index().head(10)

In [ ]:
# The rows in which the protein was not quantified in patient Con1
proteins.loc[proteins['Con1'].isna(),
             ['protein_group', 'genes', 'Con1', 'Con2', 'Con3']].head()

In [ ]:
# Option 1: keep only the proteins quantified in every patient ("complete cases")
complete = proteins.dropna(subset=sample_columns)
print("complete proteins:", complete.shape[0], "of", proteins.shape[0])

# Option 2: keep the proteins quantified in at least 70 % of the patients
keep = protein_values.notna().mean(axis=1) >= 0.7
print("proteins in >=70 % of patients:", int(keep.sum()))

# Option 3: impute, here with the lowest intensity observed anywhere in the matrix
# (a crude stand-in for "below the detection limit" - we discuss better options later)
imputed = proteins.copy()
imputed[sample_columns] = imputed[sample_columns].fillna(protein_values.min().min())
print("missing values after imputation:", int(imputed[sample_columns].isna().sum().sum()))

Compare the three numbers the cell printed. Complete cases keep only the protein groups
quantified in all 45 patients; the 70 % rule keeps a good deal more and still leaves gaps;
filling with the global minimum keeps everything and invents a large number of values, all
of them identical, which no real measurement process would ever produce. None of the three
is *the* right answer — the right answer depends on what you intend to do next, and it
belongs in your methods section either way.

This afternoon's **proteomics analysis** session takes the same decision seriously. Instead
of one global minimum it draws each replacement from a narrow normal distribution shifted
below the observed range of that sample — encoding "we did not see it, so assume it was just
under the detection limit" without pretending every unseen protein had exactly the same
intensity. Same 27 % of missing values; a defensible answer instead of a placeholder.

### ✋ Exercise 3

Three questions about the missing values themselves. The second asks for a judgement rather
than a line of code — answer it in words.

1. Using the protein matrix, compute the number of missing values per patient and per
   protein group. Is the missingness spread evenly across patients, or is there a sample
   that stands out?

```python
proteins = pd.read_csv(PROTEINS_URL, sep='\t')
sample_columns = [c for c in proteins.columns
                  if c not in ['protein_group', 'protein_names', 'genes', 'description']
                  and not c.startswith("QC")]
```

2. Assign the value **0** to the missing values in the column **Con1**. Why is 0 a poor
   choice for an intensity — and what would it do to a log₂ transformation and to a group
   mean?

3. Replace the missing values of each protein group with the median intensity of that
   protein across the patients (hint: `.median(axis=1)` together with `fillna`), and compare
   how many protein groups you keep with this approach against dropping every protein group
   that has any missing value. Which of the two would you defend to a reviewer, given what
   you now know about why the values are missing?

## 🔗 Where this goes next

Next, we visualise these same tables with **Plotly** — and the first
thing that session does is `melt` a matrix into long format, because that is the shape a
plotting library wants a table to be in. Sections 7 and 8 are its prerequisite.

After the **proteomics analysis** session takes the protein matrix through the whole
chain — transpose, log₂, filter on completeness, impute, normalise, test, interpret — and
every one of those steps is a pandas operation from this notebook with a reason attached.
When you meet `data = proteins_wide.set_index("protein_group")[patient_ids].T` there, or a
completeness filter written as `notna().mean() >= 0.70`, you will have written both already.

## 📚 Further reading

- [pandas documentation](https://pandas.pydata.org/pandas-docs/stable/) — the user guide
  chapters *Reshaping and pivot tables* and *Merge, join, concatenate* are the two worth
  reading end to end.
- [pandas cheat sheet](https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis/blob/main/cheat_sheets/Pandas_Cheat_Sheet.pdf)
  — one printable page in `cheat_sheets/`; keep it next to the keyboard for the first week.
- [Course repository](https://github.com/Multiomics-Analytics-Group/course_multi-omics_analysis)
  — every table used above, with its provenance documented in `material/datasets.md`.
- McKinney W (2022) *Python for Data Analysis*, 3rd edition — free online at
  <https://wesmckinney.com/book/>; chapters 5 to 8 cover everything in this notebook slowly
  and properly.
- Wickham H (2014) *Tidy Data.* J Stat Softw 59:10. — where "long format" comes from, and
  the clearest argument for why it is worth the extra rows.